In [ ]:
%%html
<style>
    h1 {color:darkblue}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. **Part 1 — OpenAI Python APIs**
   * 01-01. Text Generation Via the Responses API
   * 01-02. Speech Recognition, Speech Synthesis, and Closed Captions
   * 01-03. Images: Generation and Style Transfer
   * 01-04. Content Moderation
   * 01-05. **Generating Code with a Codex Model and the Responses API**
2. Part 2 — OpenAI Agents SDK
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Generating Code with a Codex Model and the Responses API
* GenAIs can write code in many programming languages
* Code quality can vary substantially
    * Well-written, functioning code that does what you ask
    * Code that does not execute or contains logic errors
* **Thoroughly test and verify any genAI-generated code**
 
---

## GPT-5.x Models
* Trained as "a true coding collaborator”
* "Excels at producing high-quality code and handling tasks such as fixing bugs, editing code, and answering questions about complex codebases"
* As programmers, we think before we write code
    * `gpt-5.5` model provides the most extensive reasoning ("thinking") capabilities

---

## Coding Demos
Covered in my Python Fundamentals videos on O'Reilly 
* Generate code from a prompt
* Explain code 
* Add type hints, docstrings, and comments to existing code
* Generate unit tests for code
* Performance-tune code
* Explain code
* Translate code between languages
* Explain the translated Java code

**OpenAI lesson should be posted by early next week**
* Will replace lesson that's there now

---

### Some Other GenAI Coding Tasks
* Generating code from diagrams/pseudocode
* Security checks
* Debugging
* Refactoring — Split code into reusable functions and classes, extract repeated code into a function, rename code elements, …
* Code reviews  —  Check for style, code smells, adherence to the latest idioms and coding guidelines, …
* and more
---

# Demos

## `import` Modules and Setup `OpenAI` Client Object

In [ ]:
import IPython
from openai import OpenAI 
from pathlib import Path
client = OpenAI() # create the OpenAI client object

<hr/>

## Generate Code from a Prompt 
* Responses API with a reasoning model

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex. Write idiomatic, readable Python code
        with type hints. Return only code.""",
    input="""Write Python code to generate a word cloud from the top 
        200 words in 'RomeoAndJuliet.txt'. Use a white background and 
        accessible colors. Create modern English and old English stop 
        words lists, including words formed by splitting contractions 
        (such as 'd', 's', and  'll'), and use those lists to eliminate
        stop words. Use the mask image 'mask_heart.png'. The required 
        files are located in the current folder's 'resources' subfolder.
        Save the image as 'RomeoAndJuliet.png' in that folder's 'outputs'
        subfolder, then display it in high resolution. Minimize use of 
        custom code. Return only the code. Use a maximum code line length
        of 74 characters.""",
    reasoning={'effort': 'low'}
)

print(response.output_text)

### Run the Code

In [ ]:
script_path = Path('resources') / 'outputs' / 'wordcloud.py'

In [ ]:
with script_path.open(mode='w') as code_file:
    code_file.write(response.output_text)

---

### File Upload -> `input_file` Pattern
1. Upload the generated code with the Files API using `purpose='user_data'`
2. Pass the returned file ID to Responses API request as an `input_file` content item

> * For code files and documents you want the model to inspect directly
> * For retrieval over larger file collections, use File Search instead of passing everything as direct file input
> * Reference: https://developers.openai.com/api/docs/guides/file-inputs


In [ ]:
# upload the file
uploaded_file = client.files.create(
    file=script_path.open(mode='rb'), 
    purpose='user_data')

In [ ]:
print(f'Uploaded file ID: {uploaded_file.id}')

### Explain File Stored on Server
* Responses API with a reasoning model
* Code can be provided in `input` prompt as text, but here we pass the file ID
* `input` object sends file ID to the model
>```python
>{
>    'type': 'input_file', 
>    'file_id': uploaded_file.id
>}
>```


In [ ]:
# use the uploaded file in a Responses API request
response = client.responses.create(
    model='gpt-5.5',
    instructions='You are Codex.',
    input=[{
        'role': 'user',
        'content': [
            {
                'type': 'input_text',
                'text': """Explain the specified file's code 
                           in detail for novice programmers."""
            },
            {
                'type': 'input_file',
                'file_id': uploaded_file.id
            }
        ]
    }],
    reasoning={'effort': 'low'}
)

IPython.display.Markdown(response.output_text)

## Adding Type Hints, Docstrings and Comments 
* [Open dice_game.py](./resources/dice_game.py)

In [ ]:
code_path = Path('resources') / 'dice_game.py'

In [ ]:
code = code_path.read_text(encoding='utf-8')

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""For the following script, add type hints, docstrings, 
        and comments using the following guidelines: 
        
        * Include a top-level docstring stating the script's purpose. 
        * Use direct comments similar to PowerPoint bullets. Do not 
          start comments with things like "We do ..."
        * Direct all comments at novice programmers so they understand
          each Python code element used and the program's logic. 
        * Always put a blank line above a full line comment. 
        * Use a maximum line length of 75 characters.
 
        DO NOT modify the original code other than adding type hints, 
        docstrings, and comments. 
        
        Return only the updated code. 

        Code:
        {code}""",
    reasoning={'effort': 'low'}
)

print(response.output_text)

## Writing Unit Tests for Code

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""Use the `doctest` module to implement unit tests for the
        functions in the following code. The updated code should provide
        a mechanism to run it in `doctest` mode with verbose output or 
        to simply run the game. Return only code, no markdown.

        Code:
        {code}""",
    reasoning={'effort': 'low'}
)

In [ ]:
print(response.output_text)

## Performance Tuning

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""Tune the following code to execute as fast as possible,
        taking advantage of all available cores and using a minimal 
        amount of code. Do not use the parallelization features from 
        the Python Standard Library. The result does not need to be 
        a list. It can be any similar indexable data structure. 
        
        In the code you return, provide a function named `profile(n)` that
        I can call to profile the performance of the original code vs. 
        your tuned code. The function should display times in seconds and
        show the speed increase between the original code and tuned code. 

        Return only code, no markdown. 

        Code:
        import random
        rolls_list = [random.randrange(1, 7) for i in range(0, 60_000_000)]
        """,
    reasoning={'effort': 'low'}
)

In [ ]:
print(response.output_text)

### Run the Tuned Code

In [ ]:
profile(n=60_000)

In [ ]:
profile(n=600_000)

In [ ]:
profile(n=60_000_000)

### Explain the Code

In [ ]:
code = response.output_text

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""Explain the following code in detail for novice programmers.

        Code:
        {code}""",
    reasoning={'effort': 'low'}
)

In [ ]:
IPython.display.Markdown(response.output_text)

## Translating Code Between Languages

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""Translate this code to idiomatic Java. For the performance
        tuned code, use libraries that auto-parallelize rather than 
        implementing the parallelized tasks yourself. Return only the 
        code. No markdown.

        Code:
        {response.output_text}
        """,
    reasoning={'effort': 'low'}
)

In [ ]:
print(response.output_text)

### Explain the Java Code

In [ ]:
code = response.output_text

In [ ]:
response = client.responses.create(
    model='gpt-5.5',
    instructions="""You are Codex.""",
    input=f"""Explain the following code in detail.

        Code:
        {code}""",
    reasoning={'effort': 'low'}
)

In [ ]:
import IPython
IPython.display.Markdown(response.output_text)

## GPT Models and Code Quality (At This Time)
* This notebook uses the Responses API with `gpt-5.5` and `reasoning={'effort': 'low'}`
* Good for
    * Correctness
    * Maintainability
    * Debugging
    * Tests
    * Security
    * Multi-step requirements matter
* Reasoning models produce better code for nontrivial tasks because they can
    * Plan
    * Inspect alternatives
    * Debug
    * Handle multi-step requirements
* Smaller `mini`/`nano` models good when speed or cost dominates
    * Quick snippets
    * Simple rewrites
    * Low-risk boilerplate
* References:
    * https://developers.openai.com/api/docs/guides/code-generation
    * https://developers.openai.com/api/docs/guides/reasoning
    * https://developers.openai.com/api/docs/models/gpt-5.5
---


## `reasoning` Effort Options (At This Time)

| Effort | When It Makes Sense |
|---|---|
| `none` | Latency-critical tasks that do not benefit from reasoning or multi-step tool use |
| `low` | Efficient reasoning for simple planning, search, tool use, coding edits, and support workflows |
| `medium` | Default starting point for `gpt-5.5`; good balance for coding, research, spreadsheets, slides, and multi-step judgement |
| `high` | Complex debugging, deep planning, agentic coding, and high-value tasks where quality matters more than latency |
| `xhigh` | Asynchronous, difficult, or long-running agentic tasks where evals justify extra latency and cost |

* Options vary by model; `gpt-5.5` supports `none`, `low`, `medium`, `high`, and `xhigh`
* Higher effort can improve quality, but increases latency and output-token cost 
    * Reasoning tokens are billed as output tokens
* References:
    * https://developers.openai.com/api/docs/guides/reasoning
    * https://developers.openai.com/api/docs/models/gpt-5.5
---


## This Notebook vs. the Codex App Workflow

| Approach | Best For | Tradeoffs |
|---|---|---|
| Responses API + coding model | Embedding code generation in your own Python apps, notebooks, services, graders, demos, and repeatable pipelines | You manage prompts, files, execution, tests, retries, storage, and UI |
| Codex App | Working directly in real projects with repo context, file edits, terminal output, sandboxing, approvals, worktrees, Git tools, browser/computer use, and iterative feedback | Less like a single API call; better for interactive software-engineering work than for embedding in another application |

* Use this notebook's API pattern when your program needs to generate, explain, transform, or review code as one step in a larger workflow
* Use the Codex App when you want an agent to work with an actual project, make edits, run commands, inspect failures, and help you review the diff
* References:
    * https://developers.openai.com/api/docs/guides/code-generation
    * https://developers.openai.com/codex/app/features

#
---
&copy; 2026 by Deitel & Associates, Inc. All Rights Reserved. 